In [37]:
# 1. Import libraries
import os
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np

from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense
from tensorflow.keras.models import Model

In [38]:

# 2. Dataset path
dataset_path = r"C:\Users\Admin\Downloads\LP IV Updated Datasets\LP IV Updated Datasets\Object Detection\caltech-101"

In [39]:
# 3. Load dataset
class_names_list = sorted([
    name for name in os.listdir(dataset_path)
    if os.path.isdir(os.path.join(dataset_path, name))
    and name != "BACKGROUND_Google"
])

In [40]:
train_data = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    image_size=(224, 224),
    batch_size=32,
    validation_split=0.2,
    subset="training",
    seed=42,
    class_names=class_names_list
)

test_data = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    image_size=(224, 224),
    batch_size=32,
    validation_split=0.2,
    subset="validation",
    seed=42,
    class_names=class_names_list
)

class_names = train_data.class_names
print("Classes:", class_names)

Found 8677 files belonging to 101 classes.
Using 6942 files for training.
Found 8677 files belonging to 101 classes.
Using 1735 files for validation.
Classes: ['Faces', 'Faces_easy', 'Leopards', 'Motorbikes', 'accordion', 'airplanes', 'anchor', 'ant', 'barrel', 'bass', 'beaver', 'binocular', 'bonsai', 'brain', 'brontosaurus', 'buddha', 'butterfly', 'camera', 'cannon', 'car_side', 'ceiling_fan', 'cellphone', 'chair', 'chandelier', 'cougar_body', 'cougar_face', 'crab', 'crayfish', 'crocodile', 'crocodile_head', 'cup', 'dalmatian', 'dollar_bill', 'dolphin', 'dragonfly', 'electric_guitar', 'elephant', 'emu', 'euphonium', 'ewer', 'ferry', 'flamingo', 'flamingo_head', 'garfield', 'gerenuk', 'gramophone', 'grand_piano', 'hawksbill', 'headphone', 'hedgehog', 'helicopter', 'ibis', 'inline_skate', 'joshua_tree', 'kangaroo', 'ketch', 'lamp', 'laptop', 'llama', 'lobster', 'lotus', 'mandolin', 'mayfly', 'menorah', 'metronome', 'minaret', 'nautilus', 'octopus', 'okapi', 'pagoda', 'panda', 'pigeon', 

In [41]:
# 4. Preprocess images
def preprocess(images, labels):
    images = preprocess_input(images)
    return images, labels

train_data = train_data.map(preprocess)
test_data = test_data.map(preprocess)

In [42]:
# 5. Load pretrained VGG16
base_model = VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze base layers
base_model.trainable = False

In [45]:
# 6. Add custom classification layers
x = GlobalAveragePooling2D()(base_model.output)
x = Dense(128, activation="relu")(x)
output = Dense(len(class_names), activation="softmax")(x)

model = Model(base_model.input, output)

In [46]:
# 7. Compile model
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
# 8. Train model
history = model.fit(
    train_data,
    epochs=2,
    validation_data=test_data
)

Epoch 1/2


C:\Users\Admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


 17/217 ━━━━━━━━━━━━━━━━━━━━ 20:01 6s/step - accuracy: 0.2261 - loss: 6.4258

In [ ]:
# 9. Evaluate model
loss, accuracy = model.evaluate(test_data)
print("Test Accuracy:", accuracy)

In [ ]:
# 10. Fine-tune model
base_model.trainable = True

for layer in base_model.layers:
    if not layer.name.startswith("block5"):
        layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.00001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.fit(
    train_data,
    epochs=1,
    validation_data=test_data
)

In [ ]:
# 11. Final evaluation
loss, accuracy = model.evaluate(test_data)
print("Final Accuracy:", accuracy)

In [ ]:
# 12. Plot accuracy
plt.plot(history.history["accuracy"], label="Training")
plt.plot(history.history["val_accuracy"], label="Validation")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

In [ ]:
# 13. Display predictions
for images, labels in test_data.take(1):

    predictions = model.predict(images)

    plt.figure(figsize=(10, 6))

    for i in range(6):
        img = images[i].numpy()
        img = img + [103.939, 116.779, 123.68]
        img = img[..., ::-1]
        img = np.clip(img, 0, 255).astype("uint8")

        plt.subplot(2, 3, i + 1)
        plt.imshow(img)

        actual = class_names[labels[i].numpy()]
        predicted = class_names[np.argmax(predictions[i])]

        plt.title("Actual: " + actual + "\nPredicted: " + predicted)
        plt.axis("off")

    plt.tight_layout()
    plt.show()